## 🏗️ LeetCode 57: Insert Interval
---

<div style="padding: 15px; border-left: 8px solid #FF5722; background-color: #fbe9e7; color: #bf360c; border-radius: 4px;">
    <strong>The Core Insight:</strong> The existing intervals are already sorted and non-overlapping. Process them in three phases: (1) intervals entirely before the new one — add directly. (2) Intervals overlapping the new one — merge all into one. (3) Intervals entirely after — add directly. Three sequential while loops, each handling one phase.
</div>

### 🛠️ The Mathematical Model

For existing interval `[s, e]` and new interval `[ns, ne]`:
- Before: `e < ns` — no overlap, add `[s, e]`
- Overlap: `s <= ne` — merge: `ns = min(ns,s)`, `ne = max(ne,e)`
- After: `s > ne` — no overlap, add remaining

---

### 📋 Problem

You are given a sorted, non-overlapping array of intervals and a new interval. Insert the new interval (merging if necessary) and return the result.

**Example 1:**
```
Input:  intervals = [[1,2],[3,5],[6,7],[8,10],[12,16]], newInterval = [4,8]
Output: [[1,2],[3,10],[12,16]]
```

**Example 2:**
```
Input:  intervals = [[1,5]], newInterval = [2,3]
Output: [[1,5]]
```

**Constraints:** 0 ≤ intervals.length ≤ 10⁴ | intervals sorted by start, non-overlapping | 0 ≤ start_i ≤ end_i ≤ 10⁵

### 🧠 Choose Your Mental Model

<table style="width:100%; border-collapse: collapse;">
    <tr style="background-color: #f2f2f2; text-align: left;">
        <th style="padding: 12px; border: 1px solid #ddd;">Model</th>
        <th style="padding: 12px; border: 1px solid #ddd;">The "Story"</th>
        <th style="padding: 12px; border: 1px solid #ddd;">Mechanism</th>
    </tr>
    <tr>
        <td style="padding: 12px; border: 1px solid #ddd;"><b>Three-Phase Scan</b></td>
        <td style="padding: 12px; border: 1px solid #ddd;">"I'm walking along a sorted timeline. First, I pass intervals that end before my new one starts — copy them. Then, I hit intervals that overlap my new one — merge them all into one big interval. Finally, I copy everything after."</td>
        <td style="padding: 12px; border: 1px solid #ddd;">Three sequential while loops: before, merge, after. Linear O(n).</td>
    </tr>
    <tr>
        <td style="padding: 12px; border: 1px solid #ddd;"><b>Maintenance Blackout Insert</b></td>
        <td style="padding: 12px; border: 1px solid #ddd;">"Adding a new maintenance window to a sorted schedule. Windows before it stay. Windows that overlap with it get absorbed. Windows after it stay."</td>
        <td style="padding: 12px; border: 1px solid #ddd;">Three clear phases, each handled by one while loop condition</td>
    </tr>
</table>

---

### ⚠️ Performance Warning

<div style="padding: 10px; border: 1px solid #ffe58f; background-color: #fffbe6; border-radius: 4px;">
    <strong>Note:</strong> Appending the new interval and re-running merge is O(n log n) — the sort is unnecessary since intervals are already sorted. Three-phase scan is O(n) — no sort needed.
</div>

## 🐢 Approach 1: Brute Force — $O(n \log n)$

In [ ]:
def insert_brute(intervals, newInterval):
    """
    Brute Force: Append new interval and re-run merge
    Time: O(n log n) — re-sorts even though input was already sorted
    """
    intervals = intervals + [newInterval]
    intervals.sort(key=lambda x: x[0])
    merged = [intervals[0]]
    for start, end in intervals[1:]:
        if start <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], end)
        else:
            merged.append([start, end])
    return merged


print(insert_brute([[1,2],[3,5],[6,7],[8,10],[12,16]], [4,8]))  # Expected: [[1,2],[3,10],[12,16]]

## 🔬 Complexity Analysis: $O(n \log n)$ vs. $O(n)$

<div style="padding: 15px; border-left: 8px solid #2196F3; background-color: #f0f7ff; color: #0d47a1; border-radius: 4px;">
    <strong>The Core Insight:</strong> The input is already sorted. We don't need to sort again. The three-phase scan exploits the pre-sorted order: scan past intervals before the new one, merge all overlapping ones into the new interval, then copy the rest. One linear pass = O(n).
</div>

---

## 📉 Why Brute Force Fails: The $O(n \log n)$ Trap

Re-sorting throws away the pre-sorted property of the input — wasted work.

| Input Type | Brute Force Performance | Reason |
|------------|------------------------|--------|
| **Already sorted input** | O(n log n) | Sort runs even though input is sorted |
| **n = 10,000** | ~130,000 sort ops | All wasted — could be 10,000 with linear scan |

---

## 🚀 The Optimal Approach: $O(n)$

Three sequential while loops, each with a clear stopping condition:
1. **Phase 1:** `intervals[i][1] < new[0]` — interval ends before new starts → add to result
2. **Phase 2:** `intervals[i][0] <= new[1]` — interval overlaps new → expand new's boundaries
3. **Phase 3:** Remaining intervals after the merge — add all directly

### The Key Lifecycle Rule
1. **Copy all intervals that end before new starts** (`end < new_start`)
2. **Merge all overlapping intervals into new** (`start <= new_end`)
3. **Append new, then copy remaining intervals**

---

## ✅ Mathematical Proof

Each interval is visited exactly once by exactly one of the three while loops. Three loops together = O(n) total. No comparisons are repeated.

<div style="padding: 10px; border-left: 8px solid #4CAF50; background-color: #f1f8f4; color: #1b5e20; border-radius: 4px;">
    <strong>✅ Summary:</strong> Three-phase scan is the optimal pattern for insert-into-sorted-intervals. Each phase has one clean stopping condition. O(n) total — no sort needed.
</div>

## 🚀 Approach 2: Three-Phase Linear Scan — $O(n)$
---

Instead of re-sorting, we exploit the pre-sorted order with **three sequential while loops**.

As we iterate:
1. **Loop 1:** Copy intervals that end before new interval starts
2. **Loop 2:** Merge all intervals that overlap with new interval (expand new's boundaries)
3. Append the (now merged) new interval; **Loop 3:** Copy remaining intervals

In [ ]:
def insert(intervals: list[list[int]], new: list[int]) -> list[list[int]]:
    """
    Optimal: Three-phase linear scan
    Time: O(n) | Space: O(n) for output
    """
    result = []
    i = 0
    n = len(intervals)

    # Phase 1: intervals that end before new starts — no overlap
    while i < n and intervals[i][1] < new[0]:
        result.append(intervals[i])
        i += 1

    # Phase 2: intervals that overlap with new — merge into new
    while i < n and intervals[i][0] <= new[1]:
        new[0] = min(new[0], intervals[i][0])   # expand new's start if needed
        new[1] = max(new[1], intervals[i][1])   # expand new's end if needed
        i += 1
    result.append(new)   # append the merged new interval

    # Phase 3: intervals that start after new ends — no overlap
    while i < n:
        result.append(intervals[i])
        i += 1

    return result


print("Optimal:", insert([[1,2],[3,5],[6,7],[8,10],[12,16]], [4,8]))   # [[1,2],[3,10],[12,16]]
print("Optimal:", insert([[1,5]], [2,3]))                               # [[1,5]]
print("Optimal:", insert([], [5,7]))                                    # [[5,7]]
print("Optimal:", insert([[1,5]], [6,8]))                               # [[1,5],[6,8]]

## 🎤 5 Interview Q&A

### Q1: What are the three phases of the algorithm?

**Answer:** Phase 1: copy all intervals whose end is strictly before the new interval's start — they can't overlap. Stopping condition: `intervals[i][1] < new[0]`. Phase 2: merge all intervals that overlap with new — stopping condition: `intervals[i][0] <= new[1]`. Phase 3: copy all remaining intervals — they start after the new interval ends.

---

### Q2: Why is this O(n) while the brute force is O(n log n)?

**Answer:** The input is already sorted. The optimal algorithm exploits this — no sort needed. Three sequential while loops, each scanning forward without backtracking, means each interval is visited exactly once. Total: O(n). Brute force re-sorts (O(n log n)) unnecessarily.

---

### Q3: What is the time complexity of the optimal approach and why?

**Answer:** O(n). Each of the n intervals is processed by exactly one of the three while loops — they collectively scan the array left to right without revisiting any interval. The pointer `i` only ever increments. No nested loops, no sorting = O(n).

---

### Q4: What if the new interval doesn't overlap with any existing interval?

**Answer:** Phase 2 may execute zero iterations (new interval fits in a gap — its start is after all Phase 1 intervals end and before Phase 3 intervals start). The new interval is appended at the correct sorted position between the Phase 1 and Phase 3 results. The algorithm handles this naturally.

---

### Q5: What are the edge cases to watch for?

**Answer:** (1) Empty intervals — Phase 1 and Phase 3 are skipped, Phase 2 is skipped, just append new. (2) New interval before all — Phase 1 is empty, Phase 2 may start from index 0. (3) New interval after all — Phase 1 processes all, Phase 2 is empty, new is appended at end. (4) New interval contains all existing — Phase 2 processes all, result is just [new] after merging. All handled by the three-loop structure.

## 📚 Key Terminology

| Term | Definition |
|------|------------|
| **Insert Interval** | Adding a new interval to a sorted non-overlapping list, merging with any overlapping intervals |
| **Three-Phase Scan** | Algorithm structure: before-merge-after, each phase a separate while loop with a clear stopping condition |
| **Pre-Sorted Property** | The given intervals are already sorted by start — enables O(n) insertion without re-sorting |
| **Phase 1 Condition** | `intervals[i][1] < new[0]` — interval ends before new starts, no overlap, safe to copy |
| **Phase 2 Condition** | `intervals[i][0] <= new[1]` — interval starts before new ends, overlap, must merge |
| **Expand Boundaries** | In Phase 2: `new[0] = min(new[0], intervals[i][0])`, `new[1] = max(new[1], intervals[i][1])` |
| **Non-overlapping** | Intervals in the output are guaranteed disjoint (no two share any point) |
| **In-order Processing** | The pointer `i` only increments — each interval is visited at most once |

## 💼 The Citi Narrative

**Context:** Citi's server scheduling system maintained a sorted list of approved maintenance windows. When a new unplanned maintenance was emergency-approved, it needed to be inserted into the existing schedule — merging with any windows it overlapped.

**Scenario:** Existing schedule: [[02:00-04:00], [06:00-08:00], [14:00-16:00]]. Emergency maintenance: [03:30-07:00]. Result: [[02:00-04:00] overlaps → extend, [06:00-08:00] overlaps → extend] = merged window [02:00-08:00]. Final schedule: [[02:00-08:00],[14:00-16:00]].

**How this pattern applied:** The three-phase scan found Phase 1 intervals (none here since [02:00] overlaps [03:30-07:00]), merged Phase 2 overlapping windows, and appended Phase 3 remainder. The algorithm ran in O(n) — no re-sort of the existing approved schedule.

**Impact:** Emergency maintenance insertions became instant — no recalculation of the full schedule. The system updated the maintenance calendar in real time as emergency windows were approved, preventing scheduling conflicts where two maintenance operations would have targeted the same server simultaneously.

In [ ]:
# -------------------------------------------------------
# PRACTICE: Add a meeting to a calendar. The calendar
# has existing non-overlapping meetings sorted by start time.
# If the new meeting overlaps any existing meeting, return False
# (cannot schedule it). Otherwise insert it and return the new calendar.
# -------------------------------------------------------

def addMeeting(calendar, new_meeting):
    # Check for any overlap — if overlap exists, return False
    # If no overlap, insert at correct sorted position and return new calendar
    pass


# Test
print(addMeeting([[1,3],[5,7]], [4,5]))    # Expected: [[1,3],[4,5],[5,7]] — wait, [4,5] and [5,7] touch
print(addMeeting([[1,3],[5,7]], [3,5]))    # Expected: False (overlaps [1,3] at 3, [5,7] at 5)
print(addMeeting([[1,3],[6,8]], [4,5]))    # Expected: [[1,3],[4,5],[6,8]]

## 🎯 Summary: Key Takeaways

### The Pattern
**Three-Phase Linear Scan** — Before | Merge | After, each phase a while loop with a clear stopping condition

### When to Use It
- ✅ Insert into sorted non-overlapping interval list
- ✅ Adding events/windows to a sorted calendar
- ✅ When input is guaranteed sorted — O(n), no sort needed
- ❌ **Don't use when:** Input is unsorted — sort first then use Merge Intervals (LC #56)

### Complexity
| Approach | Time | Space |
|----------|------|-------|
| Brute Force (re-sort) | $O(n \log n)$ | $O(n)$ |
| Optimal (Three-Phase) | $O(n)$ | $O(n)$ |

### Interview Confidence Checklist
- [ ] Can describe all three phases and their stopping conditions
- [ ] Can write all three while loops from memory
- [ ] Can explain why no sort is needed (pre-sorted input)
- [ ] Can trace through the edge case where new interval overlaps all existing
- [ ] Can connect to maintenance window scheduling use case

---

**"Simplicity and clarity is Gold."** — Sean's Study Mantra

Master Insert Interval and you've mastered the three-phase pattern — before, merge, after. It's clean, linear, and elegant. 🚀